In [6]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import json

# setting the seed for numpy
np.random.seed(10)

In [8]:
with open("../data_raw/unclean.json") as f:
    json_data = json.load(f)

# Convert dictionary to DataFrame
df = pd.DataFrame(json_data["data"])

# Save to CSV
df.to_csv("../data_clean/frailty_data.csv", index=False)

In [9]:
# reading back the saved csv
df = pd.read_csv("../data_clean/frailty_data.csv")
df.head()

,Height,Weight,Age,Grip_strength,Frailty
0,65.8,112,30,30,N
1,71.5,136,19,31,N
2,69.4,153,45,29,N
3,68.2,142,22,28,Y
4,67.8,144,29,24,Y


In [10]:
#converting height and weight
def inches_to_cm(x):
  return x * 2.54

def lb_to_kg(x):
  return x * 0.453592

df[["Height"]] = df[["Height"]].apply(inches_to_cm)
df[["Weight"]] = df[["Weight"]].apply(lb_to_kg)
df.head()

,Height,Weight,Age,Grip_strength,Frailty
0,167.132,50.802304,30,30,N
1,181.610,61.688512,19,31,N
2,176.276,69.399576,45,29,N
3,173.228,64.410064,22,28,Y
4,172.212,65.317248,29,24,Y


In [11]:
# Adding BMI column

#BMI = Weight_kg / (Height_m ** 2) (round to 2 decimals).

df['BMI'] = round(df['Weight'] / (df['Height'] ** 2), 2)

df.tail()

,Height,Weight,Age,Grip_strength,Frailty,BMI
5,174.498,55.791816,50,26,N,0.0
6,177.292,63.956472,51,22,Y,0.0
7,178.054,61.688512,23,20,Y,0.0
8,172.466,50.802304,17,19,N,0.0
9,169.672,54.431040,39,31,N,0.0


In [12]:
# add categorical feature age group
def age_group(x):
  if x < 30:
    return "<30"
  elif x >= 30 and x <= 45:
    return "30–45"
  elif x >= 46 and x <= 60:
    return "46–60"
  else:
    return ">60"

df['AgeGroup'] = df['Age'].apply(age_group)
df.head()

,Height,Weight,Age,Grip_strength,Frailty,BMI,AgeGroup
0,167.132,50.802304,30,30,N,0.0,30–45
1,181.610,61.688512,19,31,N,0.0,<30
2,176.276,69.399576,45,29,N,0.0,30–45
3,173.228,64.410064,22,28,Y,0.0,<30
4,172.212,65.317248,29,24,Y,0.0,<30


In [13]:
# binary encoding
df["Frailty"] = df["Frailty"].map({"Y": 1, "N": 0})
df.head()

,Height,Weight,Age,Grip_strength,Frailty,BMI,AgeGroup
0,167.132,50.802304,30,30,0,0.0,30–45
1,181.610,61.688512,19,31,0,0.0,<30
2,176.276,69.399576,45,29,0,0.0,30–45
3,173.228,64.410064,22,28,1,0.0,<30
4,172.212,65.317248,29,24,1,0.0,<30


In [14]:
# AgeGroup_<30, AgeGroup_30–45, AgeGroup_46–60, AgeGroup_>60

df['AgeGroup_<30'] = (df['AgeGroup'] == '<30').map({True: 1, False:0})
df['AgeGroup_30–45'] = (df['AgeGroup'] == '30-45').map({True: 1, False:0})
df['AgeGroup_46–60'] = (df['AgeGroup'] == '46-60').map({True: 1, False:0})
df['AgeGroup_>60'] = (df['AgeGroup'] == '>60').map({True: 1, False:0})

df.head()

,Height,Weight,Age,Grip_strength,Frailty,BMI,AgeGroup,AgeGroup_<30,AgeGroup_30–45,AgeGroup_46–60,AgeGroup_>60
0,167.132,50.802304,30,30,0,0.0,30–45,0,0,0,0
1,181.610,61.688512,19,31,0,0.0,<30,1,0,0,0
2,176.276,69.399576,45,29,0,0.0,30–45,0,0,0,0
3,173.228,64.410064,22,28,1,0.0,<30,1,0,0,0
4,172.212,65.317248,29,24,1,0.0,<30,1,0,0,0


In [15]:
# create summary table
summary_table = df.describe().T[['mean', '50%', 'std']]
summary_table.rename(columns={'50%': 'median'}, inplace=True)

summary_table.head()

,mean,median,std
Height,174.244000,173.863000,4.243481
Weight,59.828785,61.688512,6.455436
Age,32.500000,29.500000,12.860361
Grip_strength,26.000000,27.000000,4.521553
Frailty,0.400000,0.000000,0.516398


In [16]:
# Correlation between Grip_strength and Frailty_binary
correlation = df['Grip_strength'].corr(df['Frailty'])

print(correlation)

-0.4758668672668007


In [18]:
with open("../results/findings.md", "w") as f:
    f.write("# EDA & Findings\n")
    f.write("## 1. Summary Table\n")
    f.write(summary_table.to_markdown())
    f.write("\n")
    f.write("## 2. Correlation between Grip Strength and Frailty\n")
    f.write(f"Correlation between Grip_strength and Frailty_binary: {correlation:}\n")